In [3]:
# Week 4 - Day 4: Hypothesis Testing

import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path


# ============================================================
# SETUP
# ============================================================

print("=" * 60)
print("WEEK 4 - DAY 4: HYPOTHESIS TESTING")
print("=" * 60)

# Detect whether the script is being run from the repository root
# or from the Week 04/Day 4 folder.
repo_root = Path.cwd()

if (repo_root / "Checkpoint-2" / "data" / "da_sales_transactions.csv").exists():
    data_path = repo_root / "Checkpoint-2" / "data" / "da_sales_transactions.csv"
    output_dir = repo_root / "Week 04" / "Day 4"

else:
    data_path = repo_root / "../../Checkpoint-2/data/da_sales_transactions.csv"
    output_dir = repo_root

data_path = data_path.resolve()
output_dir = output_dir.resolve()

output_dir.mkdir(parents=True, exist_ok=True)


# ============================================================
# LOAD DATASET
# ============================================================

df = pd.read_csv(data_path)

print("\nDataset loaded successfully.")
print(f"Number of transactions: {len(df)}")

print("\nColumns:")
print(df.columns.tolist())


# ============================================================
# CREATE REVENUE
# ============================================================

df["Revenue"] = (
    df["Units"]
    * df["Unit_Price"]
    * (1 - df["Discount_Pct"] / 100)
)


# ============================================================
# TASK 1: BUSINESS HYPOTHESES
# ============================================================

print("\n" + "=" * 60)
print("TASK 1: BUSINESS HYPOTHESES")
print("=" * 60)

print("""
SCENARIO 1: AVERAGE TRANSACTION REVENUE

H0: The mean transaction revenue is equal to 8000.
H1: The mean transaction revenue is different from 8000.


SCENARIO 2: WEBSITE VS APP REVENUE

H0: The mean revenue from Website transactions equals App transactions.
H1: The mean revenue from Website transactions differs from App transactions.


SCENARIO 3: RETURN RATE

H0: The overall return rate is equal to 10%.
H1: The overall return rate is different from 10%.
""")


# ============================================================
# TASK 2: ONE-SAMPLE T-TEST
# ============================================================

print("=" * 60)
print("TASK 2: ONE-SAMPLE T-TEST")
print("=" * 60)

revenue = df["Revenue"].dropna()

# Hypothesized population mean
test_value = 8000

# One-sample t-test
t_stat, p_value = stats.ttest_1samp(
    revenue,
    popmean=test_value
)

# 95% confidence interval
confidence_level = 0.95

confidence_interval = stats.t.interval(
    confidence_level,
    len(revenue) - 1,
    loc=revenue.mean(),
    scale=stats.sem(revenue)
)

print(f"Sample mean revenue: {revenue.mean():.2f}")
print(f"Hypothesized mean: {test_value:.2f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")

print(
    f"95% Confidence Interval: "
    f"({confidence_interval[0]:.2f}, "
    f"{confidence_interval[1]:.2f})"
)

alpha = 0.05

if p_value < alpha:
    print("Decision: Reject H0.")
    print(
        "Conclusion: The mean transaction revenue is "
        "significantly different from 8000."
    )
else:
    print("Decision: Fail to reject H0.")
    print(
        "Conclusion: There is not enough evidence that "
        "the mean transaction revenue differs from 8000."
    )


# Save one-sample test results
one_sample_result = pd.DataFrame({
    "Test": ["One-sample t-test"],
    "Sample_Mean": [revenue.mean()],
    "Hypothesized_Mean": [test_value],
    "T_Statistic": [t_stat],
    "P_Value": [p_value],
    "CI_Lower": [confidence_interval[0]],
    "CI_Upper": [confidence_interval[1]]
})

one_sample_result.to_csv(
    output_dir / "one_sample_ttest.csv",
    index=False
)

print("\nOne-sample test results saved successfully.")


# ============================================================
# TASK 3: TWO-SAMPLE T-TEST
# ============================================================

print("\n" + "=" * 60)
print("TASK 3: TWO-SAMPLE T-TEST")
print("=" * 60)

# Website revenue
website_revenue = df.loc[
    df["Channel"] == "Website",
    "Revenue"
].dropna()

# App revenue
app_revenue = df.loc[
    df["Channel"] == "App",
    "Revenue"
].dropna()

print(f"Website transactions: {len(website_revenue)}")
print(f"App transactions: {len(app_revenue)}")

print(
    f"Website mean revenue: "
    f"{website_revenue.mean():.2f}"
)

print(
    f"App mean revenue: "
    f"{app_revenue.mean():.2f}"
)


# Welch's two-sample t-test
t_stat_two, p_value_two = stats.ttest_ind(
    website_revenue,
    app_revenue,
    equal_var=False
)


# ------------------------------------------------------------
# Confidence interval for difference between means
# ------------------------------------------------------------

mean_difference = (
    website_revenue.mean()
    - app_revenue.mean()
)

website_variance = website_revenue.var(ddof=1)
app_variance = app_revenue.var(ddof=1)

website_n = len(website_revenue)
app_n = len(app_revenue)

se_difference = np.sqrt(
    (website_variance / website_n)
    +
    (app_variance / app_n)
)


# Welch-Satterthwaite degrees of freedom

df_welch = (
    (
        (website_variance / website_n)
        +
        (app_variance / app_n)
    ) ** 2
    /
    (
        (
            (website_variance / website_n) ** 2
            /
            (website_n - 1)
        )
        +
        (
            (app_variance / app_n) ** 2
            /
            (app_n - 1)
        )
    )
)


# Critical t-value for 95% CI

critical_value = stats.t.ppf(
    0.975,
    df_welch
)


# Confidence interval

ci_lower = (
    mean_difference
    - critical_value * se_difference
)

ci_upper = (
    mean_difference
    + critical_value * se_difference
)


print(f"\nt-statistic: {t_stat_two:.4f}")
print(f"p-value: {p_value_two:.6f}")

print(
    f"95% CI for mean difference: "
    f"({ci_lower:.2f}, {ci_upper:.2f})"
)

print(
    f"Mean difference "
    f"(Website - App): {mean_difference:.2f}"
)


# Decision

if p_value_two < alpha:
    print("Decision: Reject H0.")
    print(
        "Conclusion: There is a statistically significant "
        "difference between Website and App mean revenue."
    )
else:
    print("Decision: Fail to reject H0.")
    print(
        "Conclusion: There is not enough evidence of a "
        "difference between Website and App mean revenue."
    )


# Save two-sample test results

two_sample_result = pd.DataFrame({
    "Test": ["Welch two-sample t-test"],
    "Website_Mean": [website_revenue.mean()],
    "App_Mean": [app_revenue.mean()],
    "Mean_Difference": [mean_difference],
    "T_Statistic": [t_stat_two],
    "P_Value": [p_value_two],
    "CI_Lower": [ci_lower],
    "CI_Upper": [ci_upper]
})

two_sample_result.to_csv(
    output_dir / "two_sample_ttest.csv",
    index=False
)

print("\nTwo-sample test results saved successfully.")


# ============================================================
# TASK 4: P-VALUE AND CONFIDENCE INTERVAL INTERPRETATION
# ============================================================

print("\n" + "=" * 60)
print("TASK 4: P-VALUE AND CONFIDENCE INTERVAL INTERPRETATION")
print("=" * 60)

print("""
P-VALUE INTERPRETATION:
A p-value below 0.05 provides evidence against the null hypothesis.

A p-value above 0.05 means there is not enough evidence to reject
the null hypothesis.

CONFIDENCE INTERVAL INTERPRETATION:
A 95% confidence interval gives a range of plausible values for
the population parameter.

For the Website vs App comparison, the 95% confidence interval
for the mean difference does not contain zero, supporting the
conclusion that the group means differ.
""")


# ============================================================
# TASK 5: TYPE I AND TYPE II ERRORS
# ============================================================

print("=" * 60)
print("TASK 5: TYPE I AND TYPE II ERRORS")
print("=" * 60)

print("""
TYPE I ERROR:

A Type I error occurs when we reject a true null hypothesis.

Business consequence:

If we conclude that Website and App mean revenue are different
when they are actually not different, the business may make
unnecessary changes to its marketing or channel strategy.


TYPE II ERROR:

A Type II error occurs when we fail to reject a false null hypothesis.

Business consequence:

If a real difference between Website and App revenue exists but
we fail to detect it, the business may miss an opportunity to
improve the lower-performing channel.


SIGNIFICANCE LEVEL (ALPHA): 0.05
""")


# ============================================================
# FINAL SUMMARY
# ============================================================

print("=" * 60)
print("HYPOTHESIS TESTING SUMMARY")
print("=" * 60)

print(f"""
One-sample t-test:
Sample mean revenue = {revenue.mean():.2f}
Hypothesized mean = {test_value:.2f}
p-value = {p_value:.6f}
95% CI = ({confidence_interval[0]:.2f}, {confidence_interval[1]:.2f})

Two-sample t-test:
Website mean = {website_revenue.mean():.2f}
App mean = {app_revenue.mean():.2f}
Mean difference = {mean_difference:.2f}
p-value = {p_value_two:.6f}
95% CI = ({ci_lower:.2f}, {ci_upper:.2f})
""")

print("Hypothesis testing analysis completed successfully.")

WEEK 4 - DAY 4: HYPOTHESIS TESTING

Dataset loaded successfully.
Number of transactions: 1000

Columns:
['Transaction_ID', 'Date', 'Customer_ID', 'Product_ID', 'Category', 'Units', 'Unit_Price', 'Discount_Pct', 'Channel', 'City', 'Return_Flag']

TASK 1: BUSINESS HYPOTHESES

SCENARIO 1: AVERAGE TRANSACTION REVENUE

H0: The mean transaction revenue is equal to 8000.
H1: The mean transaction revenue is different from 8000.


SCENARIO 2: WEBSITE VS APP REVENUE

H0: The mean revenue from Website transactions equals App transactions.
H1: The mean revenue from Website transactions differs from App transactions.


SCENARIO 3: RETURN RATE

H0: The overall return rate is equal to 10%.
H1: The overall return rate is different from 10%.

TASK 2: ONE-SAMPLE T-TEST
Sample mean revenue: 8021.85
Hypothesized mean: 8000.00
t-statistic: 0.1254
p-value: 0.900266
95% Confidence Interval: (7679.79, 8363.91)
Decision: Fail to reject H0.
Conclusion: There is not enough evidence that the mean transaction reve